# Notebook 06 — Visualization & Final Results

This notebook generates all publication-quality figures:
1. **Choropleth maps** — mortality, heatwave days, RSVI by region
2. **Time series** — national trends, regional comparisons
3. **Case studies** — paired regional comparisons
4. **Regression plots** — coefficient comparison, interaction effects

**Prerequisites:** Run Notebooks 04 and 05.

In [ ]:
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

from src.utils.config import load_config, get_path
from src.utils.io import load_dataframe

cfg = load_config()
fig_dir = get_path(cfg, 'figures')
fig_dir.mkdir(parents=True, exist_ok=True)

# Load data
panel = load_dataframe(get_path(cfg, 'processed_data') / 'panel_dataset.csv')
print(f"Panel loaded: {panel.shape}")

---
## 1. Choropleth Maps

In [ ]:
# Load NUTS-2 boundaries
from src.data.nuts2_boundaries import load_italy_nuts2
from src.visualization.maps import merge_panel_with_geometry, plot_choropleth

boundaries_dir = get_path(cfg, 'raw_data') / 'boundaries'

try:
    nuts2_gdf = load_italy_nuts2(boundaries_dir)
    gdf = merge_panel_with_geometry(panel, nuts2_gdf)
    print(f"Merged GeoDataFrame: {gdf.shape}")
    has_geo = True
except Exception as e:
    print(f"Could not load boundaries: {e}")
    print("Run Notebook 01 to download boundaries first.")
    has_geo = False

In [ ]:
# Map: Heatwave days in 2022
if has_geo and 'hw_days' in gdf.columns:
    gdf_2022 = gdf[gdf['year'] == 2022]
    
    plot_choropleth(
        gdf_2022, 'hw_days',
        title='Heatwave Days — Summer 2022',
        cmap='YlOrRd',
        legend_label='Number of Heatwave Days',
        output_path=fig_dir / 'heatwave_map_2022.png',
    )
    
    # Also show inline
    fig, ax = plt.subplots(figsize=(8, 10))
    gdf_2022.plot(column='hw_days', cmap='YlOrRd', legend=True,
                  edgecolor='0.3', linewidth=0.5, ax=ax,
                  legend_kwds={'label': 'Heatwave Days', 'shrink': 0.6})
    ax.set_title('Heatwave Days — Summer 2022', fontsize=14, fontweight='bold')
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()

In [ ]:
# Map: RSVI in 2022
if has_geo and 'rsvi' in gdf.columns:
    plot_choropleth(
        gdf_2022, 'rsvi',
        title='Social Vulnerability Index (RSVI) — 2022',
        cmap='PuRd',
        legend_label='RSVI (0=Low, 1=High)',
        output_path=fig_dir / 'rsvi_map_2022.png',
        vmin=0, vmax=1,
    )

In [ ]:
# Multi-year heatwave maps
if has_geo:
    from src.visualization.maps import plot_multi_year_choropleth
    
    years = sorted(panel['year'].unique())
    if 'hw_days' in gdf.columns:
        plot_multi_year_choropleth(
            gdf, 'hw_days', years,
            title='Heatwave Days Across Italy (2012–2022)',
            output_path=fig_dir / 'heatwave_map_all_years.png',
        )

---
## 2. National Trends

In [ ]:
from src.visualization.timeseries import plot_national_trends, plot_regional_comparison

plot_national_trends(panel, output_path=fig_dir / 'national_trends.png')

In [ ]:
# Regional comparison — top/bottom 5 by heatwave days
if 'hw_days' in panel.columns:
    plot_regional_comparison(
        panel, 'hw_days', top_n=5,
        output_path=fig_dir / 'regional_comparison_hw_days.png',
    )

---
## 3. Case Study Comparisons

Three paired comparisons between high- and low-vulnerability regions.

In [ ]:
from src.visualization.case_studies import plot_case_study_pair

for pair in cfg['case_studies']['pairs']:
    print(f"\n{'='*60}")
    print(f"Case Study: {pair['name']}")
    print(f"  {pair['high_vulnerability']} (high) vs {pair['low_vulnerability']} (low)")
    print(f"  Rationale: {pair['rationale']}")
    print(f"{'='*60}")
    
    safe_name = pair['name'].lower().replace(' ', '_')
    
    plot_case_study_pair(
        panel,
        region_a=pair['high_vulnerability'],
        region_b=pair['low_vulnerability'],
        pair_name=pair['name'],
        output_path=fig_dir / f'case_study_{safe_name}.png',
    )

---
## 4. Regression Result Plots

In [ ]:
from src.visualization.regression_plots import plot_coefficient_comparison, plot_interaction_effect

# Load regression results table
table_dir = get_path(cfg, 'tables')
results_path = table_dir / 'regression_results.csv'

if results_path.exists():
    results_table = load_dataframe(results_path)
    
    # Coefficient comparison across models
    plot_coefficient_comparison(
        results_table,
        output_path=fig_dir / 'coeff_comparison.png',
    )
    print("✅ Coefficient comparison plot saved.")
else:
    print("❌ No regression results found. Run Notebook 05 first.")

In [ ]:
# Interaction effect plot (marginal effect of heat at different RSVI levels)
# This requires the fitted model object — reload if available

try:
    from src.analysis.panel_regression import prepare_panel_index, run_model_h2
    
    panel_idx = prepare_panel_index(panel)
    h2 = run_model_h2(panel_idx)
    
    if h2['results'] is not None:
        plot_interaction_effect(
            panel, h2['results'],
            heat_var='hw_days',
            output_path=fig_dir / 'interaction_effect.png',
        )
        print("✅ Interaction effect plot saved.")
except Exception as e:
    print(f"Could not generate interaction plot: {e}")

---
## 5. Summary of All Outputs

In [ ]:
print("Generated Outputs:")
print("="*60)

# Figures
print("\n📊 Figures (outputs/figures/):")
for f in sorted(fig_dir.glob('*.png')):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name}  ({size_kb:.0f} KB)")

# Tables 
print(f"\n📋 Tables (outputs/tables/):")
for f in sorted(table_dir.glob('*')):
    if f.name == '.gitkeep':
        continue
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name}  ({size_kb:.0f} KB)")

print("\n✅ All visualizations complete!")
print("\nThese figures can be directly included in the final paper.")